Copyright 2025 DeepMind Technologies Limited.

In [ ]:
# @title SPDX-License-Identifier: Apache-2.0
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# RAG Tutorial with OneTwo

This tutorial demonstrates how to build a **Retrieval-Augmented Generation (RAG)** system using the OneTwo framework. RAG is a technique where we enhance an LLM's responses by providing it with relevant information retrieved from a data source.

We will cover:
1. **Basic Retrieval**: How to use a simple search or LLM to find information.
2. **Preparing Data for Search (Indexing)**: How to take a collection of documents, break them into manageable pieces (chunking), and make them searchable.
3. **Answering Questions**: How to combine retrieval with generation to answer questions based on your data.
4. **AI Agents with Search**: How to give an AI agent the ability to search for information to solve complex tasks.

This tutorial is similar in spirit to the main [tutorial.ipynb](tutorial.ipynb) but focuses specifically on the retrieval features of OneTwo.

# Setup


We will use the `GoogleGenAIAPI` backend for this tutorial. For more details on connecting to other backends (like `OpenAI`) or advanced configuration, please refer to the main `tutorial.ipynb`.

In [ ]:
model_selection = 'Gemini API: gemini-2.5-flash'

OneTwo can connect to publicly-hosted Gemini models via the Gemini API. If you have not used the Gemini API before, you will need to first create an account and API key following the instructions on https://ai.google.dev/. Then either copy-paste your API key into the text box, or store it in the 'GOOGLE_API_KEY' environment variable.

In [ ]:
# You can specify your API key either here or as an environment variable.
google_api_key = ''  # @param {type: 'string'}

if google_api_key:
  print('Using API key specified in the colab.')

if not google_api_key and running_in_colab:
  try:
    google_api_key = userdata.get('GOOGLE_API_KEY')
    print('Loaded GOOGLE_API_KEY from user secrets.')
  except:
    pass

if not google_api_key and 'GOOGLE_API_KEY' in os.environ:
  google_api_key = os.environ['GOOGLE_API_KEY']
  print('Loaded GOOGLE_API_KEY from environment.')

if not google_api_key:
  raise ValueError(
      'The GOOGLE_API_KEY must be specified either here or in a user secret or '
      'in the environment.'
  )

Using API key specified in the colab.


## Imports

In [ ]:
import dataclasses
import pprint

from google.genai import types as genai_types
from onetwo import ot
from onetwo.backends import google_genai_api
from onetwo.builtins import llm
# from onetwo.evaluation import datasets
from onetwo.stdlib.retrieval import retrieval
from onetwo.utils import colab_utils

## Caching Configuration

In [ ]:
# Here we define a location in which to store a cache of requests/replies for
# each backend of interest, which we can use for speeding up running of the
# colab if we re-run it (or make iterative modifications to it) in the future.
# We will use a separate cache file for each backend.
OWN_CACHE_DIRECTORY = '/tmp/onetwo_colab_backend_caches/tutorial'

# If you would like to share cache files with others in your working group, you
# can optionally specify another shared cache directory here. If you specify
# this, then we will read from shared cache directory and give precedence to its
# contents, while merging in any additional content from OWN_CACHE_DIRECTORY.
# When saving the cache, however, we will by default write only to
# OWN_CACHE_DIRECTORY to reduce the chance of people clobbering each other's
# changes.
SHARED_CACHE_DIRECTORY = None

# If you want to automatically merge in content from any of your teammates'
# cache directories or from a cache that was output by a batch eval run,
# you can list the additional directories here.
ADDITIONAL_CACHE_DIRECTORIES = []

In [ ]:
# We encapsulate the logic for loading/saving the caches for a set of backends
# in the `CachedBackends` class. Each time we construct a backend, we will add
# it to the `cached_backends` mapping so that we can load and save them all at
# once.
cached_backends = colab_utils.CachedBackends(
    own_cache_directory=OWN_CACHE_DIRECTORY,
    shared_cache_directory=SHARED_CACHE_DIRECTORY,
    additional_cache_directories=ADDITIONAL_CACHE_DIRECTORIES,
)

## Gemini API

In [ ]:
# E.g., 'Gemini API: gemini-1.0-pro' ==> 'gemini-1.0-pro'.
model_name = model_selection.split(':')[-1].strip()

generate_config = {}
# We are disabling safety settings for the purposes of this colab to
# avoid the chance of getting empty responses in the case of
# over-triggering of the safety controls. You can set this to an
# appropriate setting for your use case.
safety_settings = google_genai_api.SAFETY_DISABLED
generate_config['safety_settings'] = safety_settings

# For thinking models, you can control the amount of thinking budget
# using the `ThinkingConfig` below. For the purposes of this colab,
# we are disabling thinking by default, as it tends to not play well
# with the use of the `max_tokens` parameter (i.e.,
# `max_output_tokens` in the GoogleGenAI API), due to it getting
# applied to the sum of the thinking tokens and actual output
# tokens, which can often lead to empty outputs.
thinking_budget = 128
generate_config['thinking_config'] = genai_types.ThinkingConfig(
    include_thoughts=False, thinking_budget=thinking_budget
)


backend = google_genai_api.GoogleGenAIAPI(
    vertexai=False,
    api_key=google_api_key,
    temperature=0.0,
    generate_model_name=f'models/{model_name}',
    chat_model_name=f'models/{model_name}',
    threadpool_size=100,
    max_retries=3,
    cache=caching.SimpleFunctionCache(
        cache_filename=cached_backends.get_cache_path(model_name),
    ),
    generate_text_kwargs=generate_config,
    generate_object_kwargs=generate_config,
    generate_content_kwargs=generate_config,
    chat_kwargs=generate_config,
)

cached_backends[model_name] = backend
backend.register()

print(f'Gemini API backend registered: {model_name}')

Gemini API backend registered: gemini-2.5-flash


# Retriever

In OneTwo, a `Retriever` is a generic interface for any strategy that takes a
query and returns a list of relevant results (typically documents or text
passages). A retriever can be anything from a simple keyword search, a vector
database, or even a web-based search engine!

Below is an example of **Google Search** as a `Retriever`. As
seen in [tutorial.ipynb](tutorial.ipynb), one way to perform Google Search is
leveraging the tool-use capabilities of the
[Google Generative AI SDK](https://github.com/googleapis/python-genai).

In [ ]:
genai_search_tool = genai_types.Tool(google_search=genai_types.GoogleSearch())
chat_config_kwargs = {
    'system_instruction': (
        'You are an expert in Google Search. Use the Search tool for every'
        ' query the user provides.'
    ),
    'tools': [genai_search_tool],
    'max_tokens': 128,
}


@dataclasses.dataclass
class SearchRetriever(retrieval.Retriever[str, str]):
  @ot.make_executable(copy_self=False)
  async def retrieve(
      self,
      query: str,
      *,
      max_results: int | None = None,
      **kwargs,
  ) -> list[str]:
    """Search engine. Returns a relevant snippet or answer to query."""
    result = await llm.instruct(query, **chat_config_kwargs)
    return [result]

search_as_retriever = SearchRetriever()

In [ ]:
query = "What is the capital of France?"
result = ot.run(search_as_retriever(query))
print(f"Result type: {type(result)}")
print(f"Result content: {result}")

Result type: <class 'list'>
Result content: ['Paris is the capital and largest city of France. It is located on the river Seine in the center of the Île-de-France region. Paris has been the political, economic, religious, and cultural capital of France since the end of the 12th century.']


# Loading HotpotQA

[HotpotQA](https://hotpotqa.github.io/) is a multi-hop question answering dataset where each question requires reasoning over multiple Wikipedia paragraphs to arrive at the answer.

OneTwo provides `HotpotQADatasetLoader`, a convenient wrapper that loads HotpotQA via TFDS and returns:

- **Examples**: Dictionaries with `question` and `answer` keys, compatible with `ot.evaluate`.
- **Documents**: `Document` objects (with `doc_id`, `title`, and `content` fields) representing the Wikipedia paragraphs associated with each example, suitable for building a retrieval corpus.

Note that Documents should not be expected to be in a particular order matching the order of the examples.

## Create the Loader

We create a `HotpotQADatasetLoader` with `max_number_of_examples=5` to load only the first 5 examples (and their associated documents) for a quick preview.

In [ ]:
loader = datasets.HotpotQADatasetLoader(
    split='validation',
    max_number_of_examples=5,
)

## Load Examples

The `load_examples()` method returns evaluation-compatible example dictionaries. Each example contains:
- `question`: The question string.
- `answer`: The gold answer string.
- `metadata`: Additional HotpotQA-specific fields (e.g., `id`, `type`, `level`).

In [ ]:
examples = ot.run(loader.load_examples())
print(f'Loaded {len(examples)} examples.\n')

for i, example in enumerate(examples):
  print(f'--- Example {i + 1} ---')
  print(f'  Question: {example["question"]}')
  print(f'  Answer:   {example["answer"]}')
  print(f'  Metadata: {example["metadata"]}')
  print()

Loaded 5 examples.

--- Example 1 ---
  Question: William Davis Ticknor, Sr. served on the board of a chemical and biotechnology company created in what year?
  Answer:   1919
  Metadata: {'id': '5ae1faf15542997f29b3c1e3', 'type': 'bridge', 'level': 'hard', 'supporting_facts': {'sent_id': array([0, 0], dtype=int32), 'title': array([b'William Davis Ticknor, Sr.', b'Commercial Solvents Corporation'],
      dtype=object)}}

--- Example 2 ---
  Question: The 2009 Fort Hood mass shooting was committed by who?
  Answer:   Nidal Hasan
  Metadata: {'id': '5ab9339f554299131ca422bb', 'type': 'bridge', 'level': 'hard', 'supporting_facts': {'sent_id': array([0, 1], dtype=int32), 'title': array([b'Nidal Hasan', b'2009 Fort Hood shooting'], dtype=object)}}

--- Example 3 ---
  Question: A Talk is an EP by which South Korean singer and dancer?
  Answer:   Hyuna
  Metadata: {'id': '5ae7376f5542992ae0d163d2', 'type': 'bridge', 'level': 'hard', 'supporting_facts': {'sent_id': array([0, 0], dtype=int32),

## Load Documents

The `load_documents()` method returns `Document` objects extracted from the HotpotQA context paragraphs. Documents are **deduplicated** by `doc_id` (since the same Wikipedia paragraph can appear across multiple examples).

Each `Document` has:
- `doc_id`: Unique identifier (the Wikipedia paragraph title).
- `title`: The paragraph title.
- `content`: The concatenated text of the paragraph sentences.

In [ ]:
documents = ot.run(loader.load_documents())
print(f'Loaded {len(documents)} unique documents.\n')

for i, doc in enumerate(documents[:5]):
  print(f'--- Document {i + 1} ---')
  print(f'  Title:    {doc.title}')
  print(
      '  Content: '
      f' {doc.content[:200]}{"..." if len(doc.content) > 200 else ""}'
  )
  print(f'  Metadata: {doc.metadata}')

Loaded 50 unique documents.

--- Document 1 ---
  Title:    Icos
  Content:  Icos Corporation (trademark ICOS) was an American biotechnology company and the largest biotechnology company in the U.S. state of Washington, before it was sold to Eli Lilly and Company in 2007.  It ...
  Metadata: {}
--- Document 2 ---
  Title:    Commercial Solvents Corporation
  Content:  Commercial Solvents Corporation (CSC) was an American chemical and biotechnology company created in 1919.
  Metadata: {}
--- Document 3 ---
  Title:    Pascal Brandys
  Content:  Pascal Brandys (born 30 November 1958 in Roanne) is a French engineer and entrepreneur.  He is a graduate of the École Polytechnique and received his M.S. in Economic Systems from Stanford University ...
  Metadata: {}
--- Document 4 ---
  Title:    Daiichi Sankyo
  Content:  Daiichi Sankyo Company, Limited (第一三共株式会社 , Daiichi Sankyō Kabushiki-kaisha ) is a global pharmaceutical company and the second largest pharmaceutical company in Japan.  It 